In [1]:
import numpy as np, time
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier

np.random.seed(7)

# ------------------------------
# 1) Hierarchy
# ------------------------------
sections = {"A": "Agriculture", "C": "Manufacturing", "G": "Trade/Repair"}
divisions = {"A": ["A01"], "C": ["C25"], "G": ["G45", "G47"]}
classes = {"A01": ["A01.1", "A01.4"], "C25": ["C25.1", "C25.2"], "G45": ["G45.2"], "G47": ["G47.1"]}

division_of_class = {}
section_of_division = {}
for sec, divs in divisions.items():
    for d in divs:
        section_of_division[d] = sec
        for c in classes[d]:
            division_of_class[c] = d
all_classes = list(division_of_class.keys())

# ------------------------------
# 2) Stress generator
# ------------------------------
base_by_class = {
    "A01.1": ["growing wheat barley cereals crop cultivation fields",
              "arable farming cereals grain oilseeds cultivation"],
    "A01.4": ["raising cattle pigs livestock dairy farm animal production",
              "animal husbandry breeding livestock dairy cows"],
    "C25.1": ["fabrication steel structures metal frames construction components",
              "manufacture structural steel beams supports frameworks"],
    "C25.2": ["manufacture metal tanks containers reservoirs pressure vessels",
              "fabrication storage tanks steel containers liquids gases"],
    "G45.2": ["maintenance repair motor vehicles diagnostics workshop servicing",
              "car repair service mechanical maintenance vehicle diagnostics"],
    "G47.1": ["retail sale supermarket grocery store mixed consumer goods",
              "operating store selling food beverages daily products retail"],
}
common_vocab = ["service","operations","quality","customer","supply","logistics","business",
                "regional","local","international","maintenance","production","processing",
                "distribution","sales","support","planning","management"]
distractors = {
    "A": ["vehicle","garage","supermarket","steel","tank","welding"],
    "C": ["farm","cows","wheat","supermarket","grocery","retail"],
    "G": ["welding","fabrication","steel","tanks","harvesting","livestock"],
}
ood_pool = [
    "providing consulting services for various clients and industries",
    "digital platform offering analytics solutions and cloud services",
    "administrative support and general business services",
]

def gen_desc(true_cls, ambiguity=0.55, distract_prob=0.45, common_k=(3,6)):
    div = division_of_class[true_cls]
    sec = section_of_division[div]
    toks = np.random.choice(base_by_class[true_cls]).split()
    k = np.random.randint(common_k[0], common_k[1]+1)
    toks += list(np.random.choice(common_vocab, size=min(k, len(common_vocab)), replace=False))
    if np.random.rand() < distract_prob:
        other = np.random.choice([s for s in sections if s != sec])
        toks += list(np.random.choice(distractors[other], size=np.random.randint(1,3), replace=False))
    if np.random.rand() < ambiguity:
        drop_k = np.random.randint(2, min(5, len(toks)))
        drop_idx = set(np.random.choice(len(toks), size=drop_k, replace=False))
        toks = [t for i,t in enumerate(toks) if i not in drop_idx]
    np.random.shuffle(toks)
    return " ".join(toks)

def sample_stress(n=50000, label_noise=0.10, ood_frac=0.10):
    X=[]; ys=[]; yd=[]; yc=[]; is_ood=[]
    for _ in range(n):
        if np.random.rand() < ood_frac:
            desc = np.random.choice(ood_pool)
            cls = np.random.choice(all_classes)
            is_ood.append(True)
        else:
            cls = np.random.choice(all_classes)
            desc = gen_desc(cls)
            is_ood.append(False)
        if np.random.rand() < label_noise:
            cls = np.random.choice([c for c in all_classes if c != cls])
        div = division_of_class[cls]
        sec = section_of_division[div]
        X.append(desc); ys.append(sec); yd.append(div); yc.append(cls)
    return np.array(X), np.array(ys), np.array(yd), np.array(yc), np.array(is_ood)

# ------------------------------
# 3) Generate dataset + split
# ------------------------------
N = 50000
LABEL_NOISE = 0.10
OOD_FRAC = 0.10

t0 = time.time()
X_text, y_sec, y_div, y_cls, is_ood = sample_stress(n=N, label_noise=LABEL_NOISE, ood_frac=OOD_FRAC)
t_gen = time.time() - t0

X_train, X_cal, y_sec_train, y_sec_cal, y_div_train, y_div_cal, y_cls_train, y_cls_cal, is_ood_train, is_ood_cal = \
    train_test_split(X_text, y_sec, y_div, y_cls, is_ood, test_size=0.3, random_state=0, stratify=y_cls)

# ------------------------------
# 4) Train models
# ------------------------------
def make_fast_text_clf():
    return make_pipeline(
        TfidfVectorizer(ngram_range=(1,1), min_df=3, max_features=7000),
        SGDClassifier(loss="log_loss", alpha=1e-5, max_iter=1400, tol=1e-3, random_state=0)
    )

t0 = time.time()
sec_clf = make_fast_text_clf().fit(X_train, y_sec_train)
div_clf = make_fast_text_clf().fit(X_train, y_div_train)
cls_clf = make_fast_text_clf().fit(X_train, y_cls_train)
t_fit = time.time() - t0

sec_labels = sec_clf.named_steps["sgdclassifier"].classes_
div_labels = div_clf.named_steps["sgdclassifier"].classes_
cls_labels = cls_clf.named_steps["sgdclassifier"].classes_

idx_sec = {lab:i for i,lab in enumerate(sec_labels)}
idx_div = {lab:i for i,lab in enumerate(div_labels)}
idx_cls = {lab:i for i,lab in enumerate(cls_labels)}

In [ ]:
# ------------------------------
# 5) RAPS + reject calibration
# ------------------------------
K_FREE = 1
LAMBDA = 0.05
EPS_SEC = EPS_DIV = EPS_CLS = 0.1
EPS_REJECT = 0.1
MAX_LEAF_SIZE = 3

def renorm_rows(v):
    s = v.sum(axis=1, keepdims=True)
    return np.divide(v, s, out=np.full_like(v, 1.0 / v.shape[1]), where=(s > 0))

def raps_score_rowwise_argsort(P, true_idx, k_free=K_FREE, lam=LAMBDA):
    order = np.argsort(-P, axis=1)
    P_sorted = np.take_along_axis(P, order, axis=1)
    true_mask = (order == true_idx[:, None])
    rank_pos = true_mask.argmax(axis=1)   # exact argsort rank, 0-based
    r = rank_pos + 1
    cumsum = np.cumsum(P_sorted, axis=1)
    cum = cumsum[np.arange(P.shape[0]), rank_pos]
    penalty = lam * np.maximum(0, r - k_free)
    return cum + penalty

def reject_score(P_sec):
    return 1.0 - P_sec.max(axis=1)

def qthr(a, eps):
    return float(np.quantile(a, 1.0-eps, method="higher"))

# Predict probabilities on calibration in batch
t0 = time.time()
P_sec_cal = sec_clf.predict_proba(X_cal)
P_div_cal = div_clf.predict_proba(X_cal)
P_cls_cal = cls_clf.predict_proba(X_cal)
t_prob = time.time() - t0

# --- thresholds ---
true_sec_idx = np.array([idx_sec[s] for s in y_sec_cal])
raps_sec_scores = raps_score_rowwise_argsort(P_sec_cal, true_sec_idx)
tau_sec = qthr(raps_sec_scores, EPS_SEC)

# Division threshold: Mondrian-by-parent (true section)
div_scores = np.empty(len(X_cal))
for s in sections.keys():
    mask = (y_sec_cal == s)
    if not np.any(mask):
        continue
    child = divisions[s]
    child_idx = np.array([idx_div[d] for d in child], dtype=int)
    P_child = renorm_rows(P_div_cal[mask][:, child_idx])
    true_div_local = np.array([child.index(d) for d in y_div_cal[mask]], dtype=int)
    div_scores[mask] = raps_score_rowwise_argsort(P_child, true_div_local)
tau_div = qthr(div_scores, EPS_DIV)

# Class threshold: Mondrian-by-parent (true division)
cls_scores = np.empty(len(X_cal))
for d, child in classes.items():
    mask = (y_div_cal == d)
    if not np.any(mask):
        continue
    child_idx = np.array([idx_cls[c] for c in child], dtype=int)
    P_child = renorm_rows(P_cls_cal[mask][:, child_idx])
    true_cls_local = np.array([child.index(c) for c in y_cls_cal[mask]], dtype=int)
    cls_scores[mask] = raps_score_rowwise_argsort(P_child, true_cls_local)
tau_cls = qthr(cls_scores, EPS_CLS)

# Reject threshold calibrated on in-dist calibration points only
rej_scores_in = reject_score(P_sec_cal[~is_ood_cal])
tau_rej = float(np.quantile(rej_scores_in, 1.0 - EPS_REJECT, method="higher"))


[False  True False ...  True False  True]
[ True False False ... False False False]
[False False  True ... False  True False]


In [10]:
# ------------------------------
# 6) EXACT set-based metrics (vectorized)
# ------------------------------
t0 = time.time()

rej_cal = (reject_score(P_sec_cal) > tau_rej)
kept = ~rej_cal

# Section membership + size
sec_in_set = (raps_score_rowwise_argsort(P_sec_cal, true_sec_idx) <= tau_sec)
sec_membership = np.column_stack([
    (raps_score_rowwise_argsort(P_sec_cal, np.full(len(X_cal), j, dtype=int)) <= tau_sec)
    for j in range(P_sec_cal.shape[1])
])
sec_set_size = sec_membership.sum(axis=1)

# Division membership + size (Mondrian true parent)
div_in_set_mondrian = np.empty(len(X_cal), dtype=bool)
div_set_size_mondrian = np.empty(len(X_cal), dtype=int)
for s in sections.keys():
    mask = (y_sec_cal == s)
    if not np.any(mask):
        continue
    child = divisions[s]
    child_idx = np.array([idx_div[d] for d in child], dtype=int)
    P_child = renorm_rows(P_div_cal[mask][:, child_idx])
    true_div_local = np.array([child.index(d) for d in y_div_cal[mask]], dtype=int)

    div_in_set_mondrian[mask] = (raps_score_rowwise_argsort(P_child, true_div_local) <= tau_div)
    memb = np.column_stack([
        (raps_score_rowwise_argsort(P_child, np.full(P_child.shape[0], j, dtype=int)) <= tau_div)
        for j in range(P_child.shape[1])
    ])
    div_set_size_mondrian[mask] = memb.sum(axis=1)

# Class membership + size (Mondrian true parent)
cls_in_set_mondrian = np.empty(len(X_cal), dtype=bool)
cls_set_size_mondrian = np.empty(len(X_cal), dtype=int)
for d, child in classes.items():
    mask = (y_div_cal == d)
    if not np.any(mask):
        continue
    child_idx = np.array([idx_cls[c] for c in child], dtype=int)
    P_child = renorm_rows(P_cls_cal[mask][:, child_idx])
    true_cls_local = np.array([child.index(c) for c in y_cls_cal[mask]], dtype=int)

    cls_in_set_mondrian[mask] = (raps_score_rowwise_argsort(P_child, true_cls_local) <= tau_cls)
    memb = np.column_stack([
        (raps_score_rowwise_argsort(P_child, np.full(P_child.shape[0], j, dtype=int)) <= tau_cls)
        for j in range(P_child.shape[1])
    ])
    cls_set_size_mondrian[mask] = memb.sum(axis=1)

# Hierarchical propagated membership along TRUE path
div_in_set_hier = sec_in_set & div_in_set_mondrian
cls_in_set_hier = div_in_set_hier & cls_in_set_mondrian

# Output-level stats (exact for this tiny hierarchy)
cls_size_by_div = {}
for d, child in classes.items():
    child_idx = np.array([idx_cls[c] for c in child], dtype=int)
    P_child = renorm_rows(P_cls_cal[:, child_idx])
    memb = np.column_stack([
        (raps_score_rowwise_argsort(P_child, np.full(P_child.shape[0], j, dtype=int)) <= tau_cls)
        for j in range(P_child.shape[1])
    ])
    cls_size_by_div[d] = memb.sum(axis=1)

leaf_union_size = np.zeros(len(X_cal), dtype=int)
for s, divs_s in divisions.items():
    s_idx = idx_sec[s]
    s_in = sec_membership[:, s_idx]
    add = np.zeros(len(X_cal), dtype=int)
    for d in divs_s:
        add += cls_size_by_div[d]
    leaf_union_size += s_in.astype(int) * add

reported_level = np.full(len(X_cal), "class_level", dtype=object)
reported_level[leaf_union_size > MAX_LEAF_SIZE] = "division_level"
reported_level[rej_cal] = "reject"

t_exact = time.time() - t0

In [15]:
# ------------------------------
# 7) Print results
# ------------------------------
print(f"=== FULLY UNIFIED EXACT METRICS (argsort-identical RAPS), N={N:,} (cal size={len(X_cal):,}) ===")
print(f"Data generation: {t_gen:.2f}s | Fit: {t_fit:.2f}s | predict_proba: {t_prob:.2f}s | exact-metrics: {t_exact:.2f}s\n")

print("Thresholds:")
print("  tau_sec:", tau_sec)
print("  tau_div:", tau_div)
print("  tau_cls:", tau_cls)
print("  tau_rej (1-maxp):", tau_rej, "=> reject if max p_sec <", float(1 - tau_rej))
print()

print("Reject stats:")
print("  reject_rate (all cal):", float(rej_cal.mean()))
print("  reject_rate on flagged OOD cal points:", float(rej_cal[is_ood_cal].mean()))
print("  reject_rate on flagged in-dist cal points:", float(rej_cal[~is_ood_cal].mean()))
print()

def mean_if(mask, arr_bool):
    return float(arr_bool[mask].mean()) if np.any(mask) else float("nan")

def avg_if(mask, arr_num):
    return float(arr_num[mask].mean()) if np.any(mask) else float("nan")

print("Coverage (exact set membership):")
print("  Section coverage (unconditional):", float(sec_in_set.mean()))
print("  Division coverage (Mondrian, unconditional):", float(div_in_set_mondrian.mean()))
print("  Class   coverage (Mondrian, unconditional):", float(cls_in_set_mondrian.mean()))
print("  Division coverage (hierarchical propagated):", float(div_in_set_hier.mean()))
print("  Class   coverage (hierarchical propagated):", float(cls_in_set_hier.mean()))
print()

print("Coverage among KEPT (not rejected):")
print("  Section coverage | kept:", mean_if(kept, sec_in_set))
print("  Division Mondrian coverage | kept:", mean_if(kept, div_in_set_mondrian))
print("  Class   Mondrian coverage | kept:", mean_if(kept, cls_in_set_mondrian))
print("  Division hierarchical coverage | kept:", mean_if(kept, div_in_set_hier))
print("  Class   hierarchical coverage | kept:", mean_if(kept, cls_in_set_hier))
print()

print("Average set sizes (all cal / kept):")
print("  |Gamma_sec|:", float(sec_set_size.mean()), "/", avg_if(kept, sec_set_size))
print("  |Gamma_div(true-parent)|:", float(div_set_size_mondrian.mean()), "/", avg_if(kept, div_set_size_mondrian))
print("  |Gamma_cls(true-parent)|:", float(cls_set_size_mondrian.mean()), "/", avg_if(kept, cls_set_size_mondrian))
print()

print("Backtracking / output-level stats (using MAX_LEAF_SIZE=%d):" % MAX_LEAF_SIZE)
print("  P(output=reject):", float((reported_level=="reject").mean()))
print("  P(output=division_level):", float((reported_level=="division_level").mean()))
print("  P(output=class_level):", float((reported_level=="class_level").mean()))

=== FULLY UNIFIED EXACT METRICS (argsort-identical RAPS), N=50,000 (cal size=15,000) ===
Data generation: 1.26s | Fit: 0.78s | predict_proba: 0.22s | exact-metrics: 0.05s

Thresholds:
  tau_sec: 1.0148082057017125
  tau_div: 1.0
  tau_cls: 1.0
  tau_rej (1-maxp): 0.12818161391514316 => reject if max p_sec < 0.8718183860848568

Reject stats:
  reject_rate (all cal): 0.1898
  reject_rate on flagged OOD cal points: 1.0
  reject_rate on flagged in-dist cal points: 0.09997778271495224

Coverage (exact set membership):
  Section coverage (unconditional): 0.9000666666666667
  Division coverage (Mondrian, unconditional): 0.9670666666666666
  Class   coverage (Mondrian, unconditional): 0.9285333333333333
  Division coverage (hierarchical propagated): 0.8779333333333333
  Class   coverage (hierarchical propagated): 0.8466

Coverage among KEPT (not rejected):
  Section coverage | kept: 0.9259442113058504
  Division Mondrian coverage | kept: 0.9800049370525796
  Class   Mondrian coverage | kept: 0

In [ ]:
required = ["X_cal","y_sec_cal","y_div_cal","y_cls_cal","is_ood_cal",
            "sec_clf","div_clf","cls_clf",
            "sec_labels","div_labels","cls_labels",
            "idx_sec","idx_div","idx_cls",
            "tau_sec","tau_div","tau_cls","tau_rej",
            "divisions","classes"]
missing = [v for v in required if v not in globals()]
if missing:
    raise RuntimeError("Missing variables (run the unified cell first): " + ", ".join(missing))

K_FREE = 1
LAMBDA = 0.05

def renorm(v):
    s = float(np.sum(v))
    return v/s if s > 0 else np.ones_like(v)/len(v)

def raps_score_1d(p, j, k_free=K_FREE, lam=LAMBDA):
    order = np.argsort(-p)
    r = int(np.where(order == j)[0][0]) + 1
    return float(np.sum(p[order[:r]]) + lam * max(0, r - k_free))

def raps_set_1d(p, labels, tau):
    return [str(labels[j]) for j in range(len(labels)) if raps_score_1d(p, j) <= tau]

def reject_score_1d(pS):
    return float(1.0 - np.max(pS))

def predict_full_one(desc, max_leaf_size=3):
    
    # get 1 level prediction

    pS = sec_clf.predict_proba([desc])[0]
    rs = reject_score_1d(pS)

    # if empty return nothing
    if rs > tau_rej:
        return {
            "section_set": [],
            "division_sets": {},
            "class_sets": {},
            "final_output": {"type":"reject", "reason":"low section confidence", "reject_score": rs},
        }

    sec_set = raps_set_1d(pS, sec_labels, tau_sec)
    if len(sec_set) == 0:
        sec_set = [str(sec_labels[int(np.argmax(pS))])]

    # get 2 level prediction (of the children)

    pD = div_clf.predict_proba([desc])[0]
    div_sets = {}
    for s in sec_set:
        child = divisions[s]
        p_child = renorm(np.array([pD[idx_div[d]] for d in child]))
        dset = raps_set_1d(p_child, child, tau_div)
        if len(dset) == 0:
            dset = [child[int(np.argmax(p_child))]]
        div_sets[s] = dset

    # get 3 level prediction (of the children)

    pC = cls_clf.predict_proba([desc])[0]
    cls_sets = {}
    leaf = []
    for ds in div_sets.values():
        for d in ds:
            child = classes[d]
            p_child = renorm(np.array([pC[idx_cls[c]] for c in child]))
            cset = raps_set_1d(p_child, child, tau_cls)
            if len(cset) == 0:
                cset = [child[int(np.argmax(p_child))]]
            cls_sets[d] = cset
            leaf.extend(cset)

    leaf = sorted(set(leaf))
    if len(leaf) <= max_leaf_size:
        final = {"type":"class_level", "codes": leaf}
    else:
        div_out = sorted({d for ds in div_sets.values() for d in ds})
        final = {"type":"division_level", "codes": div_out}

    return {
        "section_set": sec_set,
        "division_sets": div_sets,
        "class_sets": cls_sets,
        "final_output": final,
    }

# Print ALL examples
for i in range(len(X_cal)):
    desc = X_cal[i]
    ts, td, tc = y_sec_cal[i], y_div_cal[i], y_cls_cal[i]
    pred = predict_full_one(desc, max_leaf_size=3)

    print("\n==================================================")
    print("Description:\n ", desc)
    print(f"True: section={ts}, division={td}, class={tc}")
    print("Section set:", pred["section_set"])
    print("Division sets:", pred["division_sets"])
    print("Class sets:", pred["class_sets"])
    print("Final output:", pred["final_output"])


Description:
  selling retail processing beverages food business local regional daily products tank customer
True: section=C, division=C25, class=C25.2
Section set: []
Division sets: {}
Class sets: {}
Final output: {'type': 'reject', 'reason': 'low section confidence', 'reject_score': 0.15266960678329178}
sec_set
['A']

Description:
  breeding animal husbandry logistics cows production international distribution dairy
True: section=A, division=A01, class=A01.4
Section set: ['A']
Division sets: {'A': ['A01']}
Class sets: {'A01': ['A01.4']}
Final output: {'type': 'class_level', 'codes': ['A01.4']}
sec_set
['G']

Description:
  beverages selling regional support store grocery retail operating retail daily products
True: section=G, division=G47, class=G47.1
Section set: ['G']
Division sets: {'G': ['G47']}
Class sets: {'G47': ['G47.1']}
Final output: {'type': 'class_level', 'codes': ['G47.1']}
sec_set
['C']

Description:
  frameworks beams supply supports manufacture steel regional structu

In [18]:
X_cal

array(['selling retail processing beverages food business local regional daily products tank customer',
       'breeding animal husbandry logistics cows production international distribution dairy',
       'beverages selling regional support store grocery retail operating retail daily products',
       ...,
       'production farm planning pigs animal cattle livestock production dairy quality management support',
       'administrative support and general business services',
       'cereals customer cultivation wheat fields crop barley'],
      dtype='<U154')